In [ ]:
# ==============================================================================
# Symbolic Frequency Response & Impulse Response via Partial Fractions
# Difference Equation:
# y[n] - (1/6)y[n-1] - (1/6)y[n-2] = x[n]
# Target Publication: Springer
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display, Markdown

sp.init_printing(use_unicode=True)

# ------------------------------------------------------------------------------
# 1. Symbolic Definitions
# ------------------------------------------------------------------------------

omega = sp.Symbol('omega', real=True)
n = sp.Symbol('n', integer=True, nonnegative=True)
x = sp.Symbol('x')
H = sp.Function('H')
h = sp.Function('h')
u = sp.Function('u')

b = [sp.Rational(1, 1)]
a = [sp.Rational(1, 1), -sp.Rational(1, 6), -sp.Rational(1, 6)]

# ------------------------------------------------------------------------------
# 2. Symbolic Frequency Response
# ------------------------------------------------------------------------------

num_sym = sum(b[i] * x**i for i in range(len(b)))
den_sym = sum(a[i] * x**i for i in range(len(a)))
H_x = sp.cancel(num_sym / den_sym)
H_omega = sp.simplify(H_x.subs(x, sp.exp(-sp.I * omega)))

display(Markdown("### **Frequency Response**"))
display(sp.Eq(H(sp.exp(sp.I * omega)), H_omega))

# ------------------------------------------------------------------------------
# 3. Symbolic Partial Fraction Expansion
# ------------------------------------------------------------------------------

partial_fracs_x = sp.apart(H_x, x)
partial_fracs_omega = sp.simplify(partial_fracs_x.subs(x, sp.exp(-sp.I * omega)))

display(Markdown("### **Partial Fraction Expansion**"))
display(sp.Eq(H(sp.exp(sp.I * omega)), partial_fracs_omega))

# ------------------------------------------------------------------------------
# 4. Symbolic Poles and Partial-Fraction Coefficients
# ------------------------------------------------------------------------------

denominator = sp.denom(H_x)
poles_x = sp.solve(sp.Eq(denominator, 0), x)

partial_terms = []

for r in poles_x:
    A = sp.simplify(-sp.residue(H_x, x, r) / r)
    p = sp.simplify(1 / r)
    partial_terms.append((A, p))

# ------------------------------------------------------------------------------
# 5. Automatic Inverse DTFT Mapping
#
# A/(1-p*e^(-jω))  <-->  A*p^n*u[n]
# ------------------------------------------------------------------------------

h_without_u = sp.simplify(sum(A * p**n for A, p in partial_terms))

display(Markdown("### **Impulse Response**"))
display(sp.Eq(h(n), h_without_u * u(n)))

# ------------------------------------------------------------------------------
# 6. Numerical Frequency Response for Plotting Only
# ------------------------------------------------------------------------------

omega_vals = np.linspace(-np.pi, np.pi, 2048)
H_function = sp.lambdify(omega, H_omega, 'numpy')
H_vals = H_function(omega_vals)

magnitude = np.abs(H_vals)
phase = np.unwrap(np.angle(H_vals))

# ------------------------------------------------------------------------------
# 7. Numerical Impulse Response for Plotting Only
# ------------------------------------------------------------------------------

n_vals = np.arange(0, 21)
h_vals = np.zeros(len(n_vals), dtype=float)

for A, p in partial_terms:
    A_num = float(sp.N(A))
    p_num = float(sp.N(p))
    h_vals += A_num * np.power(p_num, n_vals)

# ------------------------------------------------------------------------------
# 8. Plotting
# ------------------------------------------------------------------------------

plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'xtick.labelsize': 12, 'ytick.labelsize': 12, 'figure.figsize': (10, 10)})

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=False)

ax1.plot(omega_vals / np.pi, magnitude, 'b-', linewidth=2)
ax1.set_title('Frequency Response - Magnitude')
ax1.set_ylabel(r'$|\mathcal{H}(e^{j\omega})|$')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.set_xlim(-1, 1)

ax2.plot(omega_vals / np.pi, phase, 'r-', linewidth=2)
ax2.set_title('Frequency Response - Phase')
ax2.set_ylabel(r'$\angle \mathcal{H}(e^{j\omega})$ (rad)')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.set_xlim(-1, 1)

ax3.stem(n_vals, h_vals, basefmt="k-", linefmt='g-', markerfmt='go')
ax3.set_title('Discrete Impulse Response $h[n]$')
ax3.set_xlabel('Time Index ($n$)')
ax3.set_ylabel('$h[n]$')
ax3.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()